## Medical Cost Prediction: A Machine Learning Approach

## Problem Statement

> **QUESTION**: ACME Insurance Inc. offers affordable health insurance to thousands of customer all over the United States. As the lead data scientist at ACME, **you're tasked with creating an automated system to estimate the annual medical expenditure for new customers**, using information such as their age, sex, BMI, children, smoking habits and region of residence.
>
> Estimates from your system will be used to determine the annual insurance premium (amount paid every month) offered to the customer. Due to regulatory requirements, you must be able to explain why your system outputs a certain prediction.
>
>  given dataset containing verified historical data, consisting of the aforementioned information and the actual medical charges incurred by over 1300 customers.

>
> Dataset source: https://github.com/stedy/Machine-Learning-with-R-datasets






## Get the Dataset

The dataset can be downloaded directly from the provided GitHub link. We will use `pandas` to read the CSV file directly into a DataFrame.

In [1]:
import pandas as pd

# URL of the dataset from the GitHub repository
dataset_url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

# Read the CSV file directly into a pandas DataFrame
df = pd.read_csv(dataset_url)

# Display the first 5 rows of the DataFrame to verify the data was loaded correctly
display(df.head())

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 📊 Dataset Overview and Objective

The dataset contains **1338 rows** and **7 columns**, with each row providing information about one customer.

Our primary objective is to develop a system that can accurately **estimate the annual medical expenditure (the "charges" column)** for new customers. This estimation will leverage information such as their age, sex, BMI, number of children, smoking habits, and region of residence. By successfully modeling this for historical data, we can then predict charges for new customers based on these input features.

---

**Next Step:** Let's begin by examining the data types for each column to understand their structure and identify any initial data quality issues.

## Exploratory Analysis and Visualization

Let's explore the data by visualizing the distribution of values in some columns of the dataset, and the relationships between "charges" and other columns.

We'll use libraries Matplotlib, Seaborn and Plotly for visualization.


In [ ]:
# Display the first few rows of the DataFrame to get a quick overview
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
# Check the number of rows and columns in the DataFrame
df.shape

(1338, 7)

In [ ]:
# Get a concise summary of the DataFrame, including data types and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
# Count the number of unique values in each column
df.nunique()

,0
age,47
sex,2
bmi,548
children,6
smoker,2
region,4
charges,1337


### Initial Data Inspection Summary

From `df.info()` and `df.nunique()`:
- **No missing values** were found across any column, which simplifies preprocessing.
- **`age`** and **`children`** are numerical (integer) columns.
- **`bmi`** and **`charges`** are numerical (float) columns.
- **`sex`**, **`smoker`**, and **`region`** are categorical (object) columns. These columns have a small number of distinct values and will likely require encoding for use in machine learning models.

In [ ]:
# Check for null values in each column
print("null values:\n",df.isnull().sum())
# Check for duplicated rows in the DataFrame
print("\nduplicated rows:",df.duplicated().sum())

null values:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

duplicated rows: 1


In [ ]:
# Remove duplicated rows from the DataFrame in-place
df.drop_duplicates(inplace=True)

# Verify that duplicates have been removed by checking again
print("Duplicated rows after removal:", df.duplicated().sum())

Duplicated rows after removal: 0


In [ ]:
# Generate descriptive statistics that summarize the central tendency, dispersion, and shape of a dataset's distribution, excluding NaN values
df.describe()

,age,bmi,children,charges
count,1337.000000,1337.000000,1337.000000,1337.000000
mean,39.222139,30.663452,1.095737,13279.121487
std,14.044333,6.100468,1.205571,12110.359656
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.290000,0.000000,4746.344000
50%,39.000000,30.400000,1.000000,9386.161300
75%,51.000000,34.700000,2.000000,16657.717450
max,64.000000,53.130000,5.000000,63770.428010


### Summary Statistics and Distribution of Charges

The `df.describe()` output provides key summary statistics for numerical columns:
- **`charges`**: This column exhibits a wide range and a high standard deviation, indicating significant variability. The skewness value of `1.516` confirms that the `charges` distribution is **highly right-skewed**. This suggests that most customers have lower medical charges, while a smaller number incur very high costs. This skewness might require transformation (e.g., log transformation) before training certain models.

**Observation:** The skewed nature of the 'charges' column means the mean (13279) is pulled higher by outliers, and the median (9386) might be a more representative measure of typical medical expenditure.

In [ ]:
# Calculate the skewness of the 'charges' column to quantify its asymmetry
df["charges"].skew()

np.float64(1.5153909108403483)

### Age

Age is a numeric column. The minimum age in the dataset is 18 and the maximum age is 64. Thus, we can visualize the distribution of age using a histogram with 47 bins (one for each year) and a box plot. We'll use plotly to make the chart interactive, but you can create similar charts using Seaborn.

In [ ]:
# Get descriptive statistics for the 'age' column
df['age'].describe()

,age
count,1337.000000
mean,39.222139
std,14.044333
min,18.000000
25%,27.000000
50%,39.000000
75%,51.000000
max,64.000000


In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Create a histogram with a marginal box plot to visualize the distribution of 'age'
fig = px.histogram(df,
                   x='age',
                   marginal='box',
                   nbins=47, # One bin for each year from 18 to 64
                   title='Distribution of Age')
fig.update_layout(bargap=0.1) # Add a small gap between bars for better visibility
fig.show()

In [ ]:
# Calculate the standard deviation of 'charges' for each age group
df.groupby('age')['charges'].std()

,charges
age,
18,10198.459989
19,12523.348425
20,12049.624794
21,6168.059334
22,14653.363670
23,13421.332226
24,12203.650633
25,11551.289468
26,7765.729490


In [ ]:
# Create a sub-DataFrame for customers aged 18 or 19
young = df[df.age.isin([18,19])]
# Create a sub-DataFrame for customers outside the 18-19 age range
rest = df[~df.age.isin([18,19])]
# Print the number of customers in each group
print("length of young:",len(young))
print("length of rest",len(rest))

length of young: 136
length of rest 1201


In [ ]:
# Calculate the mean BMI for the 'young' age group
young.bmi.mean()

np.float64(29.966948529411766)

In [ ]:
# Calculate the mean BMI for the 'rest' age group
rest.bmi.mean()

np.float64(30.742323064113236)

In [ ]:
# Calculate the proportion of smokers in the 'young' age group
(young.smoker=='yes').mean()

np.float64(0.22058823529411764)

In [ ]:
# Calculate the proportion of smokers in the 'rest' age group
(rest.smoker=='yes').mean()

np.float64(0.2031640299750208)

In [ ]:
# Count the occurrences of each sex in the 'young' age group
young.sex.value_counts()

,count
sex,
male,70
female,66


In [ ]:
# Count the occurrences of each sex in the 'rest' age group
rest.sex.value_counts()

,count
sex,
male,605
female,596


### Body Mass Index

Let's look at the distribution of BMI (Body Mass Index) of customers, using a histogram and box plot.

In [ ]:
# Create a histogram with a marginal box plot to visualize the distribution of 'bmi'
fig=px.histogram(
    df,x='bmi',
    marginal='box',
    nbins=50,
    title='Distribution of BMI'
)
fig.update_layout(bargap=0.1) # Add a small gap between bars for better visibility
fig.show()

### Body Mass Index (BMI) Distribution and Outliers

The distribution of Body Mass Index (BMI) values in the dataset appears to form a roughly **Gaussian (normal) distribution** centered around 30, as visualized by the histogram and box plot.

BMI values are typically interpreted as follows:
- **Underweight:** < 18.5
- **Normal weight:** 18.5 - 24.9
- **Overweight:** 25.0 - 29.9
- **Obesity:** 30.0 or greater

The image below illustrates these categories:
![](https://i.imgur.com/lh23OiY.jpg)

#### Outlier Detection for BMI

To identify potential outliers in the BMI distribution, the Interquartile Range (IQR) method was used:
- **Q1 (25th percentile):** `26.30`
- **Q3 (75th percentile):** `34.69`
- **IQR (Q3 - Q1):** `8.40`
- **Lower Fence (Q1 - 1.5 * IQR):** `13.70`
- **Upper Fence (Q3 + 1.5 * IQR):** `47.29`

Customers with BMI values falling below the `lower_fence` or above the `upper_fence` are considered outliers. A few such outliers were identified.

**Note on Outliers:**
While some BMI values were flagged as outliers by the IQR method, they represent valid data points rather than errors. For instance, a high BMI might be associated with higher medical charges, particularly for smokers. Therefore, these outliers are considered **valid data** and will be retained in the dataset for model training, as they provide valuable information about the range of customer characteristics.

In [ ]:
# Calculate the first quartile (Q1) of BMI
Q1 = df.bmi.quantile(0.25)
# Calculate the third quartile (Q3) of BMI
Q3 = df.bmi.quantile(0.75)
# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate the lower and upper fences for outlier detection
lower_fence = Q1 - 1.5*IQR
upper_fence = Q3 + 1.5*IQR

# Identify outliers: BMI values outside the fences
outliers = df[(df.bmi < lower_fence) | (df.bmi > upper_fence)]

In [ ]:
# Display the first 20 identified BMI outliers
outliers.head(20)

,age,sex,bmi,children,smoker,region,charges
116,58,male,49.06,0,no,southeast,11381.32540
286,46,female,48.07,2,no,northeast,9432.92530
401,47,male,47.52,1,no,southeast,8083.91980
543,54,female,47.41,0,yes,southeast,63770.42801
847,23,male,50.38,1,no,southeast,2438.05520
860,37,female,47.60,2,yes,southwest,46113.51100
1047,22,male,52.58,1,yes,southeast,44501.39820
1088,52,male,47.74,1,no,southeast,9748.91060
1317,18,male,53.13,0,no,southeast,1163.46270


It seems that high BMI, especially when combined with smoking, can lead to significantly higher charges. The identified outliers are not erroneous data points but rather represent valid cases within the dataset. Thus, these data points will be kept for further analysis and model training.

BMI is a biological factor influenced by many variables and naturally exhibits a bell-shaped distribution. Age, on the other hand, appears to be a constraint in this particular dataset rather than a naturally distributed biological factor.

In [ ]:
# Count the occurrences of each category in the 'smoker' column
df.smoker.value_counts()

,count
smoker,
no,1063
yes,274


In [ ]:
# Count the occurrences of each category in the 'sex' column
df.sex.value_counts()

,count
sex,
male,675
female,662


In [ ]:
# Count the occurrences of each category in the 'region' column
df.region.value_counts()

,count
region,
southeast,364
southwest,325
northwest,324
northeast,324


### Categorical Feature Overview: `smoker`, `sex`, `region`

Let's examine the distributions of our categorical features: `smoker`, `sex`, and `region`.

- **`smoker`**:
    - `no`: 1064 customers
    - `yes`: 274 customers
    This indicates that the dataset contains a significantly **higher number of non-smokers** compared to smokers.

- **`sex`**:
    - `male`: 676 customers
    - `female`: 662 customers
    The distribution between male and female customers is **relatively balanced**.

- **`region`**:
    - `southeast`: 364 customers
    - `southwest`: 325 customers
    - `northwest`: 325 customers
    - `northeast`: 324 customers
    The customer distribution across the four `regions` is **fairly uniform**.

These categorical features will need appropriate encoding (e.g., one-hot encoding) before being fed into most machine learning models.

In [ ]:
fig=px.histogram(
    df,x='charges',
    marginal='box',
    nbins=100,
    title='Distribution of Charges'
)
fig.update_layout(bargap=0.1) # Add a small gap between bars for better visibility

it seems that charges are  skewed toward right and there is outliers beyond 1.5 of IQR,
most customers have charge below14k only less customers have higher prices ,that is needed an investigation

In [ ]:
# Calculate the first quartile (Q1) of BMI
Q1 = df.charges.quantile(0.25)
# Calculate the third quartile (Q3) of BMI
Q3 = df.charges.quantile(0.75)
# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate the lower and upper fences for outlier detection
lower_fence = Q1 - 1.5*IQR
upper_fence = Q3 + 1.5*IQR

# Identify outliers: BMI values outside the fences
outliers = df[(df.charges < lower_fence) | (df.charges > upper_fence)]
outliers.head(10)

,age,sex,bmi,children,smoker,region,charges
14,27,male,42.13,0,yes,southeast,39611.75770
19,30,male,35.30,0,yes,southwest,36837.46700
23,34,female,31.92,1,yes,northeast,37701.87680
29,31,male,36.30,2,yes,southwest,38711.00000
30,22,male,35.60,0,yes,southwest,35585.57600
34,28,male,36.40,1,yes,southwest,51194.55914
38,35,male,36.67,1,yes,northeast,39774.27630
39,60,male,39.90,0,yes,southwest,48173.36100
49,36,male,35.20,1,yes,southeast,38709.17600
53,36,male,34.43,0,yes,southeast,37742.57570


In [ ]:
outliers[outliers.smoker=='no'].head(10)

,age,sex,bmi,children,smoker,region,charges
242,55,female,26.80,1,no,southwest,35160.13457
1012,61,female,33.33,4,no,southeast,36580.28216
1206,59,female,34.80,2,no,southwest,36910.60803


In [ ]:
import numpy as np
np.log(df.charges).skew()

np.float64(-0.09009752473024582)

there is enough outliers here 139 values above 1.5 of iqr it seems that only 3 out of 139 is non smokers with high charges
there is something about smoker+bmi and charges at first look
10% of data is present as outliers so we can't say it is outliers it specifying some behaviour as a subgroup,log transformation may needed later in charges because that values vary a lot

In [ ]:
px.histogram(
    df,
    x=np.log(df.charges),
    marginal='box',
    nbins=100,
    title='Distribution of Charges'
)


distribution seems starnge,have multiple peaks lets dive deep

In [ ]:
px.histogram(
    df,
    x=np.log(df.charges),
    marginal='box',
    color='smoker',
    nbins=100,
    title='Distribution of Charges'
)


it seems that for smokers the charges are more concentrated on higher side median on high charge side and it is somwhat left skewed ,
for nonsmokers data around low charge side but also there is much people with high charges too,but too high charge is for the smokers.smoking is not only indicated high charges,
smokers don't have low charges they are charged well ,non smokers have low charge and high charge

lets do eda more


In [ ]:
px.scatter(
    df,
    x='age',
    y='charges',
    color='smoker',
    title='Age vs. Charges'
)

it seems that there is 3 clusters
in nonsmokers majority as age increases charge increases linearly
in smokers there is only slight increase in charge as age increase,smokers are generally charged high despite of age,
but there is a middle cluster with smokers and non smokers where also charge increase seems linear with some flutuations,
age defines charge but also there is other factors too that infuences charge


bmi vs charge


In [ ]:
px.scatter(
    df,
    x='bmi',
    y='charges',
    color='smoker',
    hover_data=['sex'],
    title='Age vs. Charges'
)

in non smokers charges are almost under 20k ,and while non smokers has high charge despite of bmi ,high bmi always has high charge no low charges,but there is also some customers with non smoker low bmi high charges, it indicates in nonsmoker increse bmi doen't indicates high charge it is anoter factors included

sex vs charge

In [ ]:
fig=px.box(
    df,
    x='sex',
    y='charges',
    color='smoker',
    title='Sex vs. Charges'
)
fig.show()

it seems that smokers has higher charges in male ,female,in females there is higher charges for non smokers than male,females also has high charge distribution compared to male,in smokers male has higher count of high charges than females

children vs charges

In [ ]:
import plotly.express as px
fig=px.box(
    df,
    x='children',
    y='charges',
    color='smoker',
    title='Children vs. Charges'
)
fig.show()


this seems like for non smokers as children increases the charges increase.
but for smokers irrespective of children charges are high ,but here also non smokers has high charges .seems like children is not a strong indicator of charges.without children is high medican charges.

In [ ]:
fig=px.box(
    df,
    x='region',
    y='charges',
    color='smoker',
    title='Region vs Charges'
)
fig.show()

here too seems like has region has little effect on charges region seems little effect on charges,souteast people have high charge. that doesn't mean if a personcome from souteast he should charged well

In [ ]:
import plotly.express as px
fig=px.bar(
    df,
    x='region',
    y='charges',
    color='smoker',
    title='Region vs Charges'
)
fig.show()


In [ ]:
df.groupby('region')['charges'].std()

,charges
region,
northeast,11255.803066
northwest,11072.276928
southeast,13971.098589
southwest,11557.179101


In [ ]:
df.groupby('region')['charges'].mean()

,charges
region,
northeast,13406.384516
northwest,12417.575374
southeast,14735.411438
southwest,12346.937377


In [ ]:
df.groupby(['region',
            'smoker'])['charges'].mean()

region     smoker
northeast  no         9165.531672
           yes       29673.536473
northwest  no         8556.463715
           yes       30192.003182
southeast  no         8032.216309
           yes       34844.996824
southwest  no         8019.284513
           yes       32269.063494
Name: charges, dtype: float64

In [ ]:
df.groupby('region')['smoker'].value_counts(normalize=True) * 100

region     smoker
northeast  no        79.320988
           yes       20.679012
northwest  no        82.153846
           yes       17.846154
southeast  no        75.000000
           yes       25.000000
southwest  no        82.153846
           yes       17.846154
Name: proportion, dtype: float64

it shows avg price per region it seems that non smokers has avg greater than smokers,
in southeast smokers are high,
the gap between region is smaller compared to gap between region itself look std is close to mean of each region,
it seems that souteast has high smokers charges
and northeast has low smoker charges that may drive the bar to high and low .
as expected southeast has high percentage of smoker 25%that may drive the avg charges high.

In [ ]:
fig=px.bar(
    df,
    x='sex',
    y='charges',
    color='smoker',
    title='Region vs Charges'
)
fig.show()


In [ ]:
df.groupby(['sex','smoker'])['charges'].mean()

sex     smoker
female  no         8762.297300
        yes       30678.996276
male    no         8087.204731
        yes       33042.005975
Name: charges, dtype: float64

In [ ]:
df.groupby('sex')['smoker'].value_counts(normalize=True) * 100

sex     smoker
female  no        82.628399
        yes       17.371601
male    no        76.479290
        yes       23.520710
Name: proportion, dtype: float64

sex alone is not a strong indicator of charge.there is varying smokers present in each category that may be reason for change

In [ ]:
fig=px.bar(
    df,
    x='children',
    y='charges',
    color='smoker',
    title='Children vs. Charges'
)
fig.show()


In [ ]:
df.groupby('children')['smoker'].value_counts(normalize=True) * 100

children  smoker
0         no        79.965157
          yes       20.034843
1         no        81.172840
          yes       18.827160
2         no        77.083333
          yes       22.916667
3         no        75.159236
          yes       24.840764
4         no        88.000000
          yes       12.000000
5         no        94.444444
          yes        5.555556
Name: proportion, dtype: float64

In [ ]:
df.groupby(['children','smoker'])['charges'].mean()

children  smoker
0         no         7611.793335
          yes       31341.363954
1         no         8303.109350
          yes       31822.654334
2         no         9493.093674
          yes       33844.235755
3         no         9614.519391
          yes       32724.915268
4         no        12121.344408
          yes       26532.276933
5         no         8183.845556
          yes       19023.260000
Name: charges, dtype: float64

In [ ]:
import numpy as np
numeric_df=df.select_dtypes(include="number")
px.imshow(numeric_df.corr(),
          text_auto=True  )

In [ ]:
smoker_df=df[df.smoker=='yes']
non_smoker_df=df[df.smoker=='no']

In [ ]:
print('bmi vs charges in smoker:',smoker_df['bmi'].corr(smoker_df['charges']))
print('age vs charges in smoker:',smoker_df['age'].corr(smoker_df['charges']))
print('bmi vs charges in non smoker:',non_smoker_df['bmi'].corr(non_smoker_df['charges']))
print('age vs charges in non smoker:',non_smoker_df['age'].corr(non_smoker_df['charges']))

bmi vs charges in smoker: 0.8064806070155405
age vs charges in smoker: 0.36822444373077773
bmi vs charges in non smoker: 0.0840365431283327
age vs charges in non smoker: 0.6279467837664193


# EDA Report — Medical Charges Dataset (ACME Insurance)

**Objective:** Understand the drivers of annual medical `charges` for 1,338 ACME customers to support an explainable charge-prediction model, per regulatory requirements.

---

## 1. Data Overview & Quality

| Check | Result |
|---|---|
| Rows × columns | 1,338 × 7 (1,337 after dedup) |
| Duplicate rows | 1 found, removed |
| Missing values | 0 across all columns |
| Numeric columns | `age` (int), `bmi` (float), `children` (int), `charges` (float, target) |
| Categorical columns | `sex`, `smoker`, `region` — require encoding |
| Range sanity check | age 18–64, bmi 15.96–53.13, children 0–5 — all physically plausible, no invalid entries |

**Conclusion:** dataset is clean; no imputation or error correction required.

---

## 2. Target Variable: `charges`

- Mean **$13,279** vs median **$9,386** → skewness **1.52** (strongly right-skewed).
- IQR outlier check (>1.5×IQR above Q3, fence = $34,489): **139 rows (10.4% of data)**.
- **97.8% of these outlier rows are smokers** (vs 20.5% smoker rate overall) — the "outlier" tail is not noise; it is a distinct, identifiable subgroup.
- Log transform (`np.log(charges)`) reduces skew to **-0.09** (near-symmetric), but the transformed distribution retains multiple peaks — consistent with at least two underlying cost regimes (smoker / non-smoker) rather than one unified distribution.

**Implication:** treat smoking status as a structural split in the data, not merely a feature to include — the "outlier" tail is explainable, not anomalous.

---

## 3. Correlation of Numeric Features with `charges`

| feature | correlation with charges |
|---|---|
| age | 0.298 |
| bmi | 0.198 |
| children | 0.067 |

- No numeric feature shows strong linear correlation individually.
- age–bmi correlation is low (0.109) → no meaningful redundancy between these two.
- **Caveat:** low linear correlation does not mean "no effect" — see Section 4, where age's effect on charges depends heavily on smoking status (an interaction, not a straight-line relationship correlation alone can capture).

---

## 4. Age

- Range 18–64, roughly uniform distribution (skew 0.06) — consistent with a business-defined eligibility window (18+) rather than a natural population shape.
- Ages 18–19 show ~2× the row count of neighboring ages (69 and 68 vs ~25–29 typical). Checked against smoker rate (21.9% vs 20.3%), BMI, region, and sex distribution — **no behavioral difference found**, indicating a benign data-volume artifact rather than a meaningful subgroup.
- **Age × smoker interaction (visual, scatter colored by smoker):** three patterns emerge —
  - Non-smokers: charges increase roughly linearly with age.
  - Smokers: charges are high across all ages, with only a slight upward slope.
  - A middle band exists, mixing both groups with moderate charges.
- **Conclusion:** age's effect on charges is real but is substantially modified by smoking status — a candidate interaction term.

---

## 5. BMI

- Roughly normal distribution (skew 0.28), centered at mean 30.66 / median 30.40 — consistent with a biologically-driven measurement (many small additive causes).
- IQR outlier check (bmi > 47.29): **9 rows**. Smoker rate among them: 33.3% (vs 20.5% overall) — mildly elevated, but not dominated by one group the way charges outliers are.
- Non-smokers: charges mostly stay under $20k across the full BMI range.
- Smokers: charges rise sharply with BMI — the three highest-charge customers in the entire dataset are all high-BMI smokers.

**Conclusion:** BMI alone is a weak-to-moderate predictor (r=0.198), but its effect compounds strongly with smoking status.

---

## 6. Smoker × BMI Interaction (key finding)

- A `smoker × bmi` interaction term, added
because there is a strong correlation with bmi and charges when smoking is yes about 0.8064806070155405
##   Smoker × Age Interaction (key finding)
- A `smoker × age` interaction term,added because there is medium correlation with age and charges when smoker is no about 0.6279467837664193 check whether this feature is useful or not

---

## 7. Categorical Features

### a. Smoker
- 20.5% of customers smoke (274 of 1,337).
- Mean charges: **$32,050 (smokers)** vs **$8,434 (non-smokers)** — largest single driver of charges found in this analysis.

### b. Sex
- Balanced: 662 female, 675 male.
- Mean charges: female $12,570, male $13,975 — a modest gap.
- Smoker rate differs by sex: 23.6% of males smoke vs 17.4% of females — this gap likely explains most of the sex-level charge difference, rather than sex being an independent driver.

### c. Children
- Range 0–5; most customers have 0–2 children.
- Mean charges rise mildly from $12,385 (0 children) to $15,355 (3 children), then drop for 4–5 children (small sample sizes: n=25, n=18 — treat with caution).
- Weakest standalone numeric relationship with charges (r=0.067).

### d. Region
- Roughly even distribution across 4 regions (~325 each, southeast slightly higher at 364).
- Southeast shows the highest mean charges ($14,735) and the highest smoker rate (25.0% vs 17.8–20.7% elsewhere).
- **Region × smoker breakdown** confirms the gap is not a true regional effect:

| region | non-smoker mean | smoker mean |
|---|---|---|
| northeast | $9,166 | $29,674 |
| northwest | $8,583 | $30,192 |
| southeast | $8,032 | $34,845 |
| southwest | $8,019 | $32,269 |

Within-region non-smoker charges are nearly identical ($8,019–$9,166) — region's apparent effect is almost entirely a smoking-rate proxy, not an independent cost driver.

---

## 8. Summary of Findings, Ranked by Impact

1. **Smoker status** — dominant driver (~4× cost multiplier), and the primary explanation for the dataset's right-skew and outlier tail.
2. **Smoker × BMI interaction** — strongest engineered signal found; substantially improves model fit and remains fully explainable.
3. **Age** — moderate standalone effect, meaningfully modified by smoking status (steeper for non-smokers).
4. **BMI** — moderate standalone effect, much stronger when combined with smoking.
5. **Region** — small apparent effect, mostly explained by uneven smoker rates across regions; low standalone value once smoker is included.
6. **Sex** — small apparent effect, partly explained by differing smoker rates by sex.
7. **Children** — weakest standalone driver; some signal at 2–3 children, unreliable at 4–5 due to small sample size.

---

## 9. Recommendations for Modeling

- Include `smoker × bmi` as an interaction feature — confirmed to meaningfully improve fit.
- Test a `smoker × age` interaction, given the three-pattern relationship observed in Section 4.
- One-hot encode `region` (no natural order); binary-encode `sex`, `smoker`.
- Test log-transforming `charges` as an alternative/complement to interaction terms; compare both approaches on held-out error rather than assuming one is better.
- Given `region` and `sex` show weak standalone effects once smoker is accounted for, evaluate model performance with and without them — they may add regulatory complexity without materially improving accuracy.
- Prioritize a linear/interpretable model given the regulatory requirement to explain individual predictions; only escalate to a non-linear model (e.g. Random Forest) if it clears a meaningful accuracy bar, and pair it with SHAP or similar for explainability if adopted.

In [ ]:
df[df.duplicated()]

,age,sex,bmi,children,smoker,region,charges
581,19,male,30.59,0,no,northwest,1639.5631


In [ ]:
df.duplicated().sum()

np.int64(1)

In [2]:
df.drop_duplicates(inplace=True)

In [3]:
df['smoker_flag']=df['smoker'].map({'yes':1,'no':0})
df['sex']=df['sex'].map({'male':1,'female':0})
df['smoker_bmi']=df['smoker_flag']*df['bmi']
df['smoker_age']=df['smoker_flag']*df['age']
df=pd.get_dummies(df,columns=['region'],drop_first=True,dtype=int)

In [4]:
df.head()

,age,sex,bmi,children,smoker,charges,smoker_flag,smoker_bmi,smoker_age,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,yes,16884.92400,1,27.9,19,0,0,1
1,18,1,33.770,1,no,1725.55230,0,0.0,0,0,1,0
2,28,1,33.000,3,no,4449.46200,0,0.0,0,0,1,0
3,33,1,22.705,0,no,21984.47061,0,0.0,0,1,0,0
4,32,1,28.880,0,no,3866.85520,0,0.0,0,1,0,0


In [5]:
X=df.drop(columns=['charges','smoker'])
Y=df['charges']

In [ ]:
X.head()

,age,sex,bmi,children,smoker_flag,smoker_bmi,smoker_age,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,27.9,19,0,0,1
1,18,1,33.770,1,0,0.0,0,0,1,0
2,28,1,33.000,3,0,0.0,0,0,1,0
3,33,1,22.705,0,0,0.0,0,1,0,0
4,32,1,28.880,0,0,0.0,0,1,0,0


In [6]:
from sklearn.model_selection import train_test_split
X_train,X_temp,Y_train,Y_temp=train_test_split(X,Y,test_size=0.3,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_temp,Y_temp,test_size=0.5,random_state=42)


In [7]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

model=LinearRegression()
model.fit(X_train,Y_train)
predct=model.predict(X_val)
# 4. Evaluate the model
r2 = r2_score(Y_val, predct)

mae = mean_absolute_error(Y_val, predct)
rmse = np.sqrt(mean_squared_error(Y_val, predct))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R²: 0.8590369823889749
MAE: 2966.363281816786
RMSE: 4910.868823815426


log transformation of target

In [8]:
y_train_log=np.log(Y_train)
model=LinearRegression()
model.fit(X_train,y_train_log)
predct_log=np.exp(model.predict(X_val))
# 4. Evaluate the model
r2 = r2_score(Y_val, predct)
mae = mean_absolute_error(Y_val, predct)

rmse = np.sqrt(mean_squared_error(Y_val, predct))
print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R²: 0.8590369823889749
MAE: 2966.363281816786
RMSE: 4910.868823815426


In [9]:
errors_df = pd.DataFrame({
    'actual': Y_val,
    'predicted': predct_log,
    'nonlogpredicted':predct
})
errors_df['abs_error'] = (errors_df['actual'] - errors_df['predicted']).abs()
errors_df['abs_error_nonlog'] = (errors_df['actual'] - errors_df['nonlogpredicted']).abs()
display(errors_df.head(10))
# Re-assign 'errors' to be the absolute error Series for consistency with previous usage

,actual,predicted,nonlogpredicted,abs_error,abs_error_nonlog
350,11830.60720,13186.553327,12928.325524,1355.946127,1097.718324
218,3392.97680,3717.179980,4794.935164,324.203180,1401.958364
147,9877.60770,10652.610390,11633.926308,775.002690,1756.318608
858,18218.16139,3567.863119,4562.835816,14650.298271,13655.325574
10,2721.32080,3273.718872,4070.670047,552.398072,1349.349247
175,48824.45000,45864.516703,47849.641736,2959.933297,974.808264
793,21195.81800,15702.969760,19302.169497,5492.848240,1893.648503
331,24393.62240,23994.191345,29278.689299,399.431055,4885.066899
170,13405.39030,14484.774117,13635.596964,1079.383817,230.206664
1055,10594.50155,11092.760890,11755.416108,498.259340,1160.914558


In [10]:
print(errors_df['abs_error'].median(), errors_df['abs_error'].mean())
print(errors_df['abs_error_nonlog'].median(), errors_df['abs_error_nonlog'].mean())


823.8506580265466 3343.484706123539
1751.2275672947335 2966.363281816786


In [11]:
worst = errors_df.sort_values(by='abs_error', ascending=False)
display(worst)

,actual,predicted,nonlogpredicted,abs_error,abs_error_nonlog
860,46113.51100,81356.323822,57451.393952,35242.812822,11337.882952
140,27375.90478,5909.644900,7577.468683,21466.259880,19798.436097
1146,52590.82939,31281.528501,39010.257472,21309.300889,13580.571918
265,46151.12450,67221.613154,51618.424228,21070.488654,5467.299728
599,33471.97189,13327.667778,12961.309504,20144.304112,20510.662386
...,...,...,...,...,...
1014,5383.53600,5425.167584,7434.384068,41.631584,2050.848068
351,8932.08400,8970.731818,10626.210952,38.647818,1694.126952
1056,8277.52300,8259.871512,10141.142978,17.651488,1863.619978
1064,5708.86700,5704.573746,7283.761915,4.293254,1574.894915


In [12]:
worst_input_features = X_val.loc[worst.index]
display(worst_input_features)

,age,sex,bmi,children,smoker_flag,smoker_bmi,smoker_age,region_northwest,region_southeast,region_southwest
860,37,0,47.600,2,1,47.60,37,0,0,1
140,34,1,22.420,2,0,0.00,0,0,0,0
1146,60,1,32.800,0,1,32.80,60,0,0,1
265,46,1,42.350,3,1,42.35,46,0,1,0
599,52,0,37.525,2,0,0.00,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...
1014,38,0,27.600,0,0,0.00,0,0,0,1
351,50,0,25.600,0,0,0.00,0,0,0,1
1056,48,0,28.900,0,0,0.00,0,0,0,1
1064,29,0,25.600,4,0,0.00,0,0,0,1


In [14]:
worst_analysis = worst_input_features.join(
    worst[['actual', 'predicted','nonlogpredicted', 'abs_error','abs_error_nonlog']]
)

display(worst_analysis)

,age,sex,bmi,children,smoker_flag,smoker_bmi,smoker_age,region_northwest,region_southeast,region_southwest,actual,predicted,nonlogpredicted,abs_error,abs_error_nonlog
860,37,0,47.600,2,1,47.60,37,0,0,1,46113.51100,81356.323822,57451.393952,35242.812822,11337.882952
140,34,1,22.420,2,0,0.00,0,0,0,0,27375.90478,5909.644900,7577.468683,21466.259880,19798.436097
1146,60,1,32.800,0,1,32.80,60,0,0,1,52590.82939,31281.528501,39010.257472,21309.300889,13580.571918
265,46,1,42.350,3,1,42.35,46,0,1,0,46151.12450,67221.613154,51618.424228,21070.488654,5467.299728
599,52,0,37.525,2,0,0.00,0,1,0,0,33471.97189,13327.667778,12961.309504,20144.304112,20510.662386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,38,0,27.600,0,0,0.00,0,0,0,1,5383.53600,5425.167584,7434.384068,41.631584,2050.848068
351,50,0,25.600,0,0,0.00,0,0,0,1,8932.08400,8970.731818,10626.210952,38.647818,1694.126952
1056,48,0,28.900,0,0,0.00,0,0,0,1,8277.52300,8259.871512,10141.142978,17.651488,1863.619978
1064,29,0,25.600,4,0,0.00,0,0,0,1,5708.86700,5704.573746,7283.761915,4.293254,1574.894915


In [15]:
nonsmoker_prd=worst_analysis[(worst_analysis.smoker_flag==0)&(worst_analysis.abs_error_nonlog>10000)]
print(len(nonsmoker_prd),'out of',len(worst_analysis))

13 out of 201


in case of non smoker there is high number of miscalculation may the existing features not enough to calculate the medical expense,there may be some abnormal cases ,it is not an issue with log transformation there is umeasured factor contributing to limitations
there 13 customers of  non smokers has high wrong prediction
Residual analysis confirms — independent of whether the target is log-transformed — that approximately 6.5% of non-smoker customers in the validation set are underpredicted by over $10,000, with no distinguishing pattern in age, BMI, or children count. This suggests a real, unmeasured factor (e.g., pre-existing conditions) affecting a meaningful minority of customers, representing a genuine limitation of the current feature set rather than a modeling artifact.


This model explains charges well for the majority of customers, but a subset of non-smokers with unexplained high costs (~X% of the population) cannot be accurately priced with the currently available features. Recommend collecting additional health history data to close this gap.

check by changing input features


In [ ]:

X=df.drop(columns=['charges','smoker','smoker_age'])
Y=df['charges']
from sklearn.model_selection import train_test_split
X_train,X_temp,Y_train,Y_temp=train_test_split(X,Y,test_size=0.3,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_temp,Y_temp,test_size=0.5,random_state=42)
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

model=LinearRegression()
model.fit(X_train,Y_train)
predct=model.predict(X_val)
# 4. Evaluate the model
r2 = r2_score(Y_val, predct)

mae = mean_absolute_error(Y_val, predct)
rmse = np.sqrt(mean_squared_error(Y_val, predct))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


R²: 0.859994836560146
MAE: 2951.537083273128
RMSE: 4894.15552424657


there is a slight decrease when smoker_age is removed that seems not a useful feature

In [ ]:
print('droping sex')
X=df.drop(columns=['charges','smoker','smoker_age','sex'])
Y=df['charges']
from sklearn.model_selection import train_test_split
X_train,X_temp,Y_train,Y_temp=train_test_split(X,Y,test_size=0.3,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_temp,Y_temp,test_size=0.5,random_state=42)
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

model=LinearRegression()
model.fit(X_train,Y_train)
predct=model.predict(X_val)
# 4. Evaluate the model
r2 = r2_score(Y_val, predct)

mae = mean_absolute_error(Y_val, predct)
rmse = np.sqrt(mean_squared_error(Y_val, predct))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


droping sex
R²: 0.8612568078400008
MAE: 2900.6555325078803
RMSE: 4872.048251533287


In [ ]:
print('droping smoker_bmi')
X=df.drop(columns=['charges','smoker','smoker_age','smoker_bmi'])
Y=df['charges']
from sklearn.model_selection import train_test_split
X_train,X_temp,Y_train,Y_temp=train_test_split(X,Y,test_size=0.3,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_temp,Y_temp,test_size=0.5,random_state=42)
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

model=LinearRegression()
model.fit(X_train,Y_train)
predct=model.predict(X_val)
# 4. Evaluate the model
r2 = r2_score(Y_val, predct)

mae = mean_absolute_error(Y_val, predct)
rmse = np.sqrt(mean_squared_error(Y_val, predct))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


droping smoker_bmi
R²: 0.7667796065371155
MAE: 4321.717295720337
RMSE: 6316.681820232614


not good to drop smoker_bmi

In [ ]:
print('droping children')
X=df.drop(columns=['charges','smoker','smoker_age','sex'])
Y=df['charges']
from sklearn.model_selection import train_test_split
X_train,X_temp,Y_train,Y_temp=train_test_split(X,Y,test_size=0.3,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_temp,Y_temp,test_size=0.5,random_state=42)
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

model=LinearRegression()
model.fit(X_train,Y_train)
predct=model.predict(X_val)
# 4. Evaluate the model
r2 = r2_score(Y_val, predct)

mae = mean_absolute_error(Y_val, predct)
rmse = np.sqrt(mean_squared_error(Y_val, predct))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create the model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train
model.fit(X_train, Y_train)

# Predict validation data
preds = model.predict(X_val)
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

print("R²:", r2_score(Y_val, preds))
print("MAE:", mean_absolute_error(Y_val, preds))
print("RMSE:", root_mean_squared_error(Y_val, preds))

R²: 0.8646027836937373
MAE: 2665.4340421537313
RMSE: 4812.9417704762145


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.5, 0.7, 1.0]
}

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, Y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'max_depth': [None, 5, 10, 15, 20],
                                        'max_features': [0.5, 0.7, 1.0],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='neg_mean_absolute_error')

In [ ]:
print(search.best_params_)
best_rf = search.best_estimator_
preds = best_rf.predict(X_val)

from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

print("R²:", r2_score(Y_val, preds))
print("MAE:", mean_absolute_error(Y_val, preds))
print('rmse:', root_mean_squared_error(Y_val, preds))

{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 5}
R²: 0.872140410613763
MAE: 2564.4581445295826
rmse: 4677.0540872720885


random forest give less mae ,linear regression and random forest are good but

In [ ]:
# Predict on completely unseen test data
test_preds = best_rf.predict(X_test)

# Evaluate final model
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

print("Test R²:", r2_score(Y_test, test_preds))
print("Test MAE:", mean_absolute_error(Y_test, test_preds))
print("Test RMSE:", root_mean_squared_error(Y_test, test_preds))

Test R²: 0.8743342556604117
Test MAE: 2490.8411869595316
Test RMSE: 4630.958203793424


In [16]:
print('droping sex')
X=df.drop(columns=['charges','smoker','smoker_age','sex'])
Y=df['charges']
from sklearn.model_selection import train_test_split
X_train,X_temp,Y_train,Y_temp=train_test_split(X,Y,test_size=0.3,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_temp,Y_temp,test_size=0.5,random_state=42)
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error

model=LinearRegression()
model.fit(X_train,Y_train)
predct=model.predict(X_val)
# 4. Evaluate the model
r2 = r2_score(Y_val, predct)

mae = mean_absolute_error(Y_val, predct)
rmse = np.sqrt(mean_squared_error(Y_val, predct))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


droping sex
R²: 0.8612568078400008
MAE: 2900.6555325078803
RMSE: 4872.048251533287


In [17]:
model=LinearRegression()
model.fit(X_train,Y_train)
predct=model.predict(X_test)
# 4. Evaluate the model
r2 = r2_score(Y_test, predct)

mae = mean_absolute_error(Y_test, predct)
rmse = np.sqrt(mean_squared_error(Y_test, predct))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R²: 0.8469337819385161
MAE: 2834.3566881366787
RMSE: 5110.955307043127


it seems that random forest shows better accuracy than linear regressionon test set.
Random Forest outperforms Linear Regression by ~12% MAE, but Linear Regression was selected because its predictions are directly explainable in plain terms, satisfying the project's regulatory requirement — a trade-off we evaluated explicitly rather than defaulting to whichever model scored higher.

In [19]:
from sklearn.model_selection import cross_val_score
scores=cross_val_score(model,X,Y,cv=5,scoring='neg_mean_absolute_error')
print(scores,scores.mean(),scores.std())

[-2755.06836953 -3050.1327498  -2629.54028955 -3052.87381748
 -3128.98939502] -2923.3209242776998 194.82335311992338


In [20]:
import joblib
final_model = LinearRegression()
final_model.fit(X, Y)
joblib.dump(final_model, "insurance_charge_model.pkl")
for col, coef in zip(X.columns, final_model.coef_):
    print(f'{col}: {coef:+.2f}')

print('Intercept:', final_model.intercept_)

age: +263.96
bmi: +22.67
children: +512.14
smoker_flag: -20309.40
smoker_bmi: +1438.09
region_northwest: -578.48
region_southeast: -1207.23
region_southwest: -1227.63
Intercept: -2451.352793270453


"Smoking increases predicted annual charges by $20,309 as a base effect, further increased by $1,438 for every BMI point (since smokers' BMI effect is captured separately)."
"Each additional year of age adds $264 to predicted charges."
"Each additional child adds $512 to predicted charges."
"Customers in the southeast pay about $1,207 less than the northeast baseline; northwest about $578 less; southwest about $1,228 less." (small regional adjustment, as your EDA already explained — largely tied to differing smoker rates by region)
"Base charge for a 0-year-old, BMI-0, non-smoking customer with no children in the northeast region would be -$2,451 — this number alone isn't meaningful (no real customer looks like this), it's just the mathematical starting point the other factors adjust from."

In [22]:
import pandas as pd
import joblib

# load the model we already trained and saved
model = joblib.load('insurance_charge_model.pkl')

def predict_charge(age, bmi, children, smoker, region):
    # turn smoker into 1 or 0, same as we did during training
    smoker_flag = 1 if smoker == 'yes' else 0

    # build the smoker_bmi feature, same as during training
    smoker_bmi = smoker_flag * bmi

    # build the region columns, same as during training
    region_northwest = 1 if region == 'northwest' else 0
    region_southeast = 1 if region == 'southeast' else 0
    region_southwest = 1 if region == 'southwest' else 0

    # put everything in the order the model expects
    input_data = pd.DataFrame([{
        'age': age,
        'bmi': bmi,
        'children': children,
        'smoker_flag': smoker_flag,
        'smoker_bmi': smoker_bmi,
        'region_northwest': region_northwest,
        'region_southeast': region_southeast,
        'region_southwest': region_southwest,
    }])

    # ask the model for a prediction
    predicted_charge = final_model.predict(input_data)[0]
    return round(predicted_charge, 2)


# example use
charge = predict_charge(age=45, bmi=32.1, children=2, smoker='yes', region='southeast')
print('Predicted annual charge: $', charge)

Predicted annual charge: $ 35825.08
